In [ ]:
# =========================
# nb_validate
# =========================

# ---------- Parameters ----------
period = "2026-03"
run_id = "manual_test_run"

# ---------- Imports ----------
from pyspark.sql import functions as F

# ---------- Helpers ----------
def q(s: str) -> str:
    return (s or "").replace("'", "''")

def load_rules_config():
    path = "/lakehouse/default/Files/config/rules_config.json"
    try:
        raw = mssparkutils.fs.head(path, 2_000_000)
        import json
        return json.loads(raw)
    except Exception:
        return {}

def is_rule_enabled(cfg, rule_id, default=True):
    if "rules" in cfg and isinstance(cfg["rules"], dict):
        r = cfg["rules"].get(rule_id, {})
        return r.get("enabled", default)
    r = cfg.get(rule_id, {})
    return r.get("enabled", default) if isinstance(r, dict) else default

def rule_param(cfg, rule_id, key, default):
    if "rules" in cfg and isinstance(cfg["rules"], dict):
        return cfg["rules"].get(rule_id, {}).get(key, default)
    return cfg.get(rule_id, {}).get(key, default) if isinstance(cfg.get(rule_id, {}), dict) else default

def dq_result_df(base_df, check_id, check_name, check_type, severity, status, metric_value, threshold_value, comparator, total_rows, failed_rows, details_json):
    return base_df.select(
        F.lit(period).alias("period"),
        F.lit(run_id).alias("run_id"),
        F.col("dfm_id"),
        F.lit(check_id).alias("check_id"),
        F.lit(check_name).alias("check_name"),
        F.lit(check_type).alias("check_type"),
        F.lit(severity).alias("severity"),
        F.lit(status).alias("status"),
        F.lit(metric_value).cast("double").alias("metric_value"),
        F.lit(threshold_value).cast("double").alias("threshold_value"),
        F.lit(comparator).alias("comparator"),
        F.lit(total_rows).cast("bigint").alias("total_rows_evaluated"),
        F.lit(failed_rows).cast("bigint").alias("rows_failed"),
        F.lit(details_json).alias("details_json"),
        F.current_timestamp().alias("evaluated_at")
    )

def dq_exception_df(df, check_id, reason):
    return (
        df.select(
            F.lit(period).alias("period"),
            F.lit(run_id).alias("run_id"),
            F.col("dfm_id"),
            F.lit(check_id).alias("check_id"),
            F.col("source_file"),
            F.col("source_row_id"),
            F.col("policyholder_number"),
            F.col("security_code"),
            F.lit(reason).alias("failure_reason"),
            F.to_json(
                F.struct(
                    F.col("isin"),
                    F.col("bid_value_gbp"),
                    F.col("holding"),
                    F.col("fx_rate"),
                    F.col("data_quality_flags")
                )
            ).alias("details_json"),
            F.current_timestamp().alias("created_at")
        )
    )

# ---------- Load Stage 2 for run ----------
stage2 = spark.table("individual_dfm_consolidated").filter(
    (F.col("period") == period) & (F.col("run_id") == run_id)
)

if stage2.rdd.isEmpty():
    print(f"[nb_validate] No Stage 2 rows for period={period}, run_id={run_id}")
    mssparkutils.notebook.exit("NO_DATA")

cfg = load_rules_config()
map_enabled = is_rule_enabled(cfg, "MAP_001", True)
mv_enabled = is_rule_enabled(cfg, "MV_001", True)
inc_enabled = is_rule_enabled(cfg, "INC_001", True)

mv_tol = float(rule_param(cfg, "MV_001", "tolerance_pct", 0.02))

base_counts = stage2.groupBy("dfm_id").agg(F.count("*").alias("total_rows"))

dq_results_parts = []
dq_exception_parts = []

# ---------- MAP_001: identifier completeness ----------
if map_enabled:
    # Allow cash positions (id_type = "Undertaking - Specific") to have no ISIN/SEDOL
    map_fail = stage2.filter(
        F.col("policyholder_number").isNull()
        | (
            (F.col("security_code").isNull() & F.col("isin").isNull() & F.col("sedol").isNull())
            & ~F.lower(F.col("id_type")).contains("undertaking")
        )
    )

    fail_counts = map_fail.groupBy("dfm_id").agg(F.count("*").alias("failed_rows"))
    map_stats = (
        base_counts.join(fail_counts, on="dfm_id", how="left")
        .fillna(0, subset=["failed_rows"])
        .withColumn(
            "status",
            F.when(F.col("failed_rows") > 0, F.lit("fail")).otherwise(F.lit("pass"))
        )
    )

    dq_results_parts.append(
        map_stats.select(
            F.lit(period).alias("period"),
            F.lit(run_id).alias("run_id"),
            F.col("dfm_id"),
            F.lit("MAP_001").alias("check_id"),
            F.lit("Identifier Completeness").alias("check_name"),
            F.lit("row_completeness").alias("check_type"),
            F.lit("exception").alias("severity"),
            F.col("status"),
            (F.col("failed_rows") / F.col("total_rows")).cast("double").alias("metric_value"),
            F.lit(0.0).cast("double").alias("threshold_value"),
            F.lit("=").alias("comparator"),
            F.col("total_rows").cast("bigint").alias("total_rows_evaluated"),
            F.col("failed_rows").cast("bigint").alias("rows_failed"),
            F.to_json(F.struct(F.col("failed_rows"), F.col("total_rows"))).alias("details_json"),
            F.current_timestamp().alias("evaluated_at")
        )
    )

    dq_exception_parts.append(dq_exception_df(map_fail, "MAP_001", "identifier_missing"))

# ---------- MV_001: valuation coherence ----------
if mv_enabled:
    mv_eval = (
        stage2
        .filter(
            F.col("holding").isNotNull()
            & F.col("local_bid_price").isNotNull()
            & F.col("fx_rate").isNotNull()
            & (F.col("fx_rate") != 0)
            & F.col("bid_value_gbp").isNotNull()
        )
        .withColumn(
            "expected_bid_value_gbp",
            F.col("holding").cast("double")
            * F.col("local_bid_price").cast("double")
            * F.col("fx_rate").cast("double")
        )
        .withColumn(
            "pct_diff",
            F.when(
                F.abs(F.col("expected_bid_value_gbp")) > 0,
                F.abs(F.col("bid_value_gbp").cast("double") - F.col("expected_bid_value_gbp")) / F.abs(F.col("expected_bid_value_gbp"))
            ).otherwise(F.lit(None).cast("double"))
        )
    )

    mv_fail = mv_eval.filter(F.col("pct_diff") > mv_tol)

    mv_totals = mv_eval.groupBy("dfm_id").agg(F.count("*").alias("total_rows"))
    mv_fails = mv_fail.groupBy("dfm_id").agg(F.count("*").alias("failed_rows"))

    mv_stats = (
        mv_totals.join(mv_fails, on="dfm_id", how="left")
        .fillna(0, subset=["failed_rows"])
        .withColumn(
            "status",
            F.when(F.col("failed_rows") > 0, F.lit("fail")).otherwise(F.lit("pass"))
        )
    )

    dq_results_parts.append(
        mv_stats.select(
            F.lit(period).alias("period"),
            F.lit(run_id).alias("run_id"),
            F.col("dfm_id"),
            F.lit("MV_001").alias("check_id"),
            F.lit("Market Value Tolerance").alias("check_name"),
            F.lit("valuation").alias("check_type"),
            F.lit("warning").alias("severity"),
            F.col("status"),
            (F.col("failed_rows") / F.col("total_rows")).cast("double").alias("metric_value"),
            F.lit(mv_tol).cast("double").alias("threshold_value"),
            F.lit("<=").alias("comparator"),
            F.col("total_rows").cast("bigint").alias("total_rows_evaluated"),
            F.col("failed_rows").cast("bigint").alias("rows_failed"),
            F.to_json(F.struct(F.col("failed_rows"), F.col("total_rows"), F.lit(mv_tol).alias("tolerance_pct"))).alias("details_json"),
            F.current_timestamp().alias("evaluated_at")
        )
    )

    dq_exception_parts.append(dq_exception_df(mv_fail, "MV_001", "market_value_tolerance_exceeded"))

# ---------- INC_001: include flag validity ----------
if inc_enabled:
    inc_fail = stage2.filter(~F.lower(F.col("include_flag")).isin("include", "remove"))

    inc_fails = inc_fail.groupBy("dfm_id").agg(F.count("*").alias("failed_rows"))
    inc_stats = (
        base_counts.join(inc_fails, on="dfm_id", how="left")
        .fillna(0, subset=["failed_rows"])
        .withColumn(
            "status",
            F.when(F.col("failed_rows") > 0, F.lit("fail")).otherwise(F.lit("pass"))
        )
    )

    dq_results_parts.append(
        inc_stats.select(
            F.lit(period).alias("period"),
            F.lit(run_id).alias("run_id"),
            F.col("dfm_id"),
            F.lit("INC_001").alias("check_id"),
            F.lit("Include Flag Domain").alias("check_name"),
            F.lit("domain").alias("check_type"),
            F.lit("exception").alias("severity"),
            F.col("status"),
            (F.col("failed_rows") / F.col("total_rows")).cast("double").alias("metric_value"),
            F.lit(0.0).cast("double").alias("threshold_value"),
            F.lit("=").alias("comparator"),
            F.col("total_rows").cast("bigint").alias("total_rows_evaluated"),
            F.col("failed_rows").cast("bigint").alias("rows_failed"),
            F.to_json(F.struct(F.col("failed_rows"), F.col("total_rows"))).alias("details_json"),
            F.current_timestamp().alias("evaluated_at")
        )
    )

    dq_exception_parts.append(dq_exception_df(inc_fail, "INC_001", "invalid_include_flag"))

# ---------- Persist gate outputs ----------
if len(dq_results_parts) == 0:
    print("[nb_validate] No rules enabled; writing pass-through gate marker")
    fallback = (
        base_counts.select(
            F.lit(period).alias("period"),
            F.lit(run_id).alias("run_id"),
            F.col("dfm_id"),
            F.lit("CFG_000").alias("check_id"),
            F.lit("Rules Configuration").alias("check_name"),
            F.lit("config").alias("check_type"),
            F.lit("warning").alias("severity"),
            F.lit("not_evaluable").alias("status"),
            F.lit(None).cast("double").alias("metric_value"),
            F.lit(None).cast("double").alias("threshold_value"),
            F.lit(None).cast("string").alias("comparator"),
            F.col("total_rows").cast("bigint").alias("total_rows_evaluated"),
            F.lit(0).cast("bigint").alias("rows_failed"),
            F.lit('{"message":"no rules enabled"}').alias("details_json"),
            F.current_timestamp().alias("evaluated_at")
        )
    )
    fallback.write.mode("append").saveAsTable("dq_results")
    mssparkutils.notebook.exit("OK")

dq_results_df = dq_results_parts[0]
for nxt in dq_results_parts[1:]:
    dq_results_df = dq_results_df.unionByName(nxt)

dq_results_df.write.mode("append").saveAsTable("dq_results")

if len(dq_exception_parts) > 0:
    dq_exception_df_all = dq_exception_parts[0]
    for nxt in dq_exception_parts[1:]:
        dq_exception_df_all = dq_exception_df_all.unionByName(nxt)
    if not dq_exception_df_all.rdd.isEmpty():
        dq_exception_df_all.write.mode("append").saveAsTable("dq_exception_rows")

# Backward-compatible event stream for existing reporting notebook.
legacy_events = (
    dq_results_df.filter(F.lower(F.col("status")) != "pass")
    .select(
        "period",
        "run_id",
        F.current_timestamp().alias("event_time"),
        "dfm_id",
        F.lit(None).cast("string").alias("dfm_name"),
        F.lit(None).cast("string").alias("policy_id"),
        F.lit(None).cast("string").alias("security_id"),
        F.col("check_id").alias("rule_id"),
        "severity",
        "status",
        "details_json",
        F.lit(None).cast("string").alias("source_file")
    )
)
legacy_events.write.mode("append").saveAsTable("validation_events")

summary = (
    dq_results_df.groupBy("check_id", "severity", "status")
    .agg(F.sum("rows_failed").alias("rows_failed"))
    .orderBy("check_id", "status")
)
display(summary)

print(f"[nb_validate] Completed for period={period}, run_id={run_id}")
mssparkutils.notebook.exit("OK")